In [ ]:
# Step 1: Install dependencies
!pip install lightgbm tqdm scikit-learn pandas numpy

# Step 2: Import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from tqdm import tqdm
import gc, re, os

# Display setup
pd.set_option('display.max_colwidth', 150)


In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print(train_df.shape, test_df.shape)
train_df.head()

(75000, 4) (75000, 3)


,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, 12 Ounce (Pack of 6)\nValue: 72.0\nUnit: Fl Oz\n",https://m.media-amazon.com/images/I/51mo8htwTHL.jpg,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butter Cookies, 8 Ounce (Pack of 4)\nBullet Point 1: Original Butter Cookies: Classic butter cookies made...",https://m.media-amazon.com/images/I/71YtriIHAAL.jpg,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy Chicken with Rice, 1.9 Ounce (Pack of 6)\nBullet Point 1: Loaded with hearty long grain wild rice a...",https://m.media-amazon.com/images/I/51+PFEe-w-L.jpg,1.97
3,55858,"Item Name: Judee’s Blue Cheese Powder 11.25 oz - Gluten-Free and Nut-Free - Use in Seasonings and Salad Dressings - Great for Dips, Spreads and Sa...",https://m.media-amazon.com/images/I/41mu0HAToDL.jpg,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Ounce - 12 per case.\nBullet Point: kedem Sherry Cooking Wine, 12.7 Ounce - 12 per case.\nValue: 12.0\n...",https://m.media-amazon.com/images/I/41sA037+QvL.jpg,66.49


In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["catalog_content"] = train_df["catalog_content"].apply(clean_text)
test_df["catalog_content"] = test_df["catalog_content"].apply(clean_text)


In [ ]:
# Combine text for TF-IDF fitting
all_text = pd.concat([train_df["catalog_content"], test_df["catalog_content"]])

# TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words='english'
)
tfidf_features = tfidf.fit_transform(all_text)

# Dimensionality reduction
svd = TruncatedSVD(n_components=256, random_state=42)
svd_features = svd.fit_transform(tfidf_features)

X_train = svd_features[:len(train_df)]
X_test = svd_features[len(train_df):]
y_train = train_df["price"].values


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f"Fold {fold}")
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    train_set = lgb.Dataset(X_tr, label=np.log1p(y_tr))
    val_set = lgb.Dataset(X_val, label=np.log1p(y_val))

    params = {
        'objective': 'regression',
        'metric': 'mae',
        'learning_rate': 0.05,
        'num_leaves': 64,
        'min_data_in_leaf': 50,
        'feature_fraction': 0.9,
        'bagging_fraction': 0.9,
        'bagging_freq': 1,
        'verbosity': -1,
        'random_state': 42
    }

    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, val_set],
        num_boost_round=2000,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=200)]
    )

    oof_preds[val_idx] = np.expm1(model.predict(X_val))
    test_preds += np.expm1(model.predict(X_test)) / kf.n_splits

    gc.collect()

Fold 1
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	training's l1: 0.164031	valid_1's l1: 0.562574
Fold 2
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	training's l1: 0.163044	valid_1's l1: 0.555287
Fold 3
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	training's l1: 0.16326	valid_1's l1: 0.558242
Fold 4
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	training's l1: 0.164531	valid_1's l1: 0.548025
Fold 5
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	training's l1: 0.163731	valid_1's l1: 0.558401


In [ ]:
mae = mean_absolute_error(y_train, oof_preds)
smape = np.mean(
    np.abs(y_train - oof_preds) / ((np.abs(y_train) + np.abs(oof_preds)) / 2)
) * 100
print(f"Validation MAE: {mae:.4f}, SMAPE: {smape:.2f}%")


Validation MAE: 12.6829, SMAPE: 55.56%


In [ ]:
# Create submission file
submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": np.clip(test_preds, a_min=0, a_max=None)
})

output_path = "/content/test_out.csv"
submission.to_csv(output_path, index=False)
print(f"Saved submission file to: {output_path}")
submission.head()


Saved submission file to: /content/test_out.csv


,sample_id,price
0,100179,14.871962
1,245611,14.452496
2,146263,22.249263
3,95658,15.314095
4,36806,25.440842
